# CUAD: a four-step agent that builds its confidence in stages

`agent_confidence(rec)` takes one CUAD record from `dev.jsonl` and runs the LLM as an agent in four steps.
Each step answers one narrow question and gives its own confidence from 0 to 1; the final confidence combines the four.

| Step | Question the agent answers | Checked by code afterwards |
| --- | --- | --- |
| 1 | Is the answer present in the contract? | |
| 2 | Where is it? Quote the passage(s). | Is each quote really in the contract? |
| 3 | In which quarter of the contract is it: 0-25%, 25-50%, 50-75% or 75-100%? | Does that match where the quote actually sits? |
| 4 | Does the question need a semantic answer (a provision that must be read and reasoned about) or data (a date, a name, an amount)? For data: what is the value and how often does it occur in the contract? | How often does the value really occur? |

Each step is one LLM call covering all the clauses still in play, so a record costs four calls, not 41 × 4.
Step 1 looks at all 41 clauses; steps 2 to 4 only at the clauses step 1 judged present.

Gold is not used by the agent. It appears only in the last section, to see how the final confidence behaves.

Needs `LLM_BASE_URL`, `LLM_API_KEY` and `TEXT_LLM_MODEL` (or `LLM_MODEL`) in `Project_1/.env`.

In [1]:
import importlib.util, json, math, os, time, urllib.error, urllib.request
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/unified/dev.jsonl").exists())
DATA = ROOT / "data"
pd.set_option("display.max_colwidth", 70, "display.max_rows", 60, "display.width", 250)

for line in (ROOT / ".env").read_text(encoding="utf-8").splitlines():      # KEY=VALUE lines; the environment wins
    key, sep, value = line.strip().partition("=")
    if sep and key and not key.startswith("#"):
        os.environ.setdefault(key.strip(), value.strip().strip("'\""))

MODEL = os.environ.get("TEXT_LLM_MODEL") or os.environ.get("LLM_MODEL")
assert MODEL and os.environ.get("LLM_BASE_URL") and os.environ.get("LLM_API_KEY"), "fill in Project_1/.env first"
print("model:", MODEL)

model: claude-sonnet-5.5


## Helpers: loading a record, calling the LLM, locating text

Nothing agent-specific here. `load_record` follows `pages` and `doc_id` to the contract text and the 41 questions.
`ask` makes one LLM call and returns the JSON object in the reply. `locate` finds a quote in the contract, ignoring
differences in spacing and capitalisation, and returns where it starts and ends.

In [2]:
REGIONS = ["0-25%", "25-50%", "50-75%", "75-100%"]
_raw = None

def load_record(index: int = 0, doc_id: str | None = None) -> dict:
    """One CUAD record from dev.jsonl, with its contract text ("context") and its questions added."""
    global _raw
    cuad = [r for r in map(json.loads, (DATA / "unified/dev.jsonl").open(encoding="utf-8")) if r["source_dataset"] == "cuad"]
    rec = next(r for r in cuad if r["doc_id"] == doc_id) if doc_id else cuad[index]
    if _raw is None:
        _raw = {d["title"]: d["paragraphs"][0] for d in json.loads((DATA / rec["pages"][0]).read_text(encoding="utf-8"))["data"]}
    para = _raw[rec["doc_id"].split(":", 1)[1]]
    questions = {qa["id"].rsplit("__", 1)[1]: qa["question"].split("Details:", 1)[-1].strip() for qa in para["qas"]}
    return {**rec, "context": para["context"], "questions": {n: questions[n] for n in rec["schema"]["properties"]}}

def ask(prompt: str, tries: int = 8) -> dict:
    """One LLM call; returns the JSON object in the reply. Waits and retries when the gateway rate-limits."""
    body = json.dumps({"model": MODEL, "max_tokens": 16000, "messages": [{"role": "user", "content": prompt}]}).encode()
    req = urllib.request.Request(os.environ["LLM_BASE_URL"].rstrip("/") + "/chat/completions", body,
                                 {"Content-Type": "application/json", "Authorization": f"Bearer {os.environ['LLM_API_KEY']}"})
    for attempt in range(tries):
        try:
            with urllib.request.urlopen(req, timeout=600) as r:
                reply = json.load(r)["choices"][0]["message"]["content"]
            out = json.loads(reply[reply.index("{"): reply.rindex("}") + 1])
            return {str(k).strip().strip("[]").strip(): v for k, v in out.items()}
        except urllib.error.HTTPError as e:
            detail = e.read().decode(errors="replace")[:300]
            if e.code in (429, 500, 502, 503, 529) and attempt < tries - 1:
                time.sleep(20)
                continue
            raise RuntimeError(f"HTTP {e.code}: {detail}") from None

def squash(text: str):
    """Whitespace runs collapsed to one space, lower-cased, plus the original index of every kept character."""
    out, pos, in_space = [], [], False
    for i, ch in enumerate(text):
        if ch.isspace():
            if not in_space and out:
                out.append(" "); pos.append(i)
            in_space = True
        else:
            out.append(ch.casefold()[:1] or ch); pos.append(i); in_space = False
    return "".join(out), pos

def locate(span: str, squashed: str, pos: list) -> tuple[int, int] | None:
    """(start, end) character positions of the span in the contract, or None if it is not there."""
    needle = squash(span)[0].strip()
    at = squashed.find(needle) if needle else -1
    return None if at < 0 else (pos[at], pos[at + len(needle) - 1] + 1)

def region_of(start: int, length: int) -> str:
    return REGIONS[min(int(4 * start / max(length, 1)), 3)]

def conf(value) -> float:
    """A confidence from the model, cleaned to the range 0 to 1 (0 if missing or not a number)."""
    try:
        return min(max(float(value), 0.0), 1.0)
    except (TypeError, ValueError):
        return 0.0

## The four agent steps

Each step is a small function: it builds one prompt, makes one call, and returns a dictionary keyed by clause name.
A step only sees what the earlier steps produced, which is what makes this an agent pipeline and not four independent questions.

In [3]:
STEP1 = """You review a contract. For each clause below, decide only ONE thing: is there anything in the contract that answers it?
Do not quote anything yet.

Reply with ONE JSON object and nothing else:
{{"<clause name>": {{"present": true or false, "confidence": <0..1>}}, ...}}

- "present": true if the contract contains a passage a careful lawyer would highlight for this clause. Most clauses are absent from most contracts.
- "confidence": how sure you are of that yes or no, from 0 to 1.
- Include all {n} clauses.

Clauses (name: what to look for):
{clauses}

--- CONTRACT ---
{contract}
"""

def step1_is_present(rec: dict) -> dict:
    """Step 1: is the answer present? -> {clause: {"present": bool, "confidence": float}}"""
    clauses = "\n".join(f"- {n}: {q}" for n, q in rec["questions"].items())
    reply = ask(STEP1.format(n=len(rec["questions"]), clauses=clauses, contract=rec["context"]))
    out = {}
    for name in rec["questions"]:
        a = reply.get(name) if isinstance(reply.get(name), dict) else {}
        out[name] = {"present": a.get("present") is True, "confidence": conf(a.get("confidence"))}
    return out


STEP2 = """You review a contract. An earlier pass decided that the clauses below ARE present in the contract.
For each one, show WHERE: quote the passage or passages that answer it.

Reply with ONE JSON object and nothing else:
{{"<clause name>": {{"spans": ["<exact quote>", ...], "confidence": <0..1>}}, ...}}

- Copy the words of every span exactly as they are in the contract. Never paraphrase or summarise.
- A passage that runs over several lines is ONE span. Keep it whole and write line breaks and runs of spaces as a single space.
  Never cut a sentence at a line break, and never return half a sentence.
- Only passages from different places in the contract are separate spans.
- For Document Name, Parties and dates, quote only the name or the date. For other clauses, quote the whole sentence or clause.
- If, on a closer look, the contract has nothing for a clause, return {{"spans": [], "confidence": 0}}.
- "confidence": how sure you are that these are exactly the passages a careful lawyer would highlight, from 0 to 1.

Clauses (name: what to look for):
{clauses}

--- CONTRACT ---
{contract}
"""

def step2_where(rec: dict, present: list) -> dict:
    """Step 2: where is the answer? -> {clause: {"spans": [...], "confidence": float}}"""
    if not present:
        return {}
    clauses = "\n".join(f"- {n}: {rec['questions'][n]}" for n in present)
    reply = ask(STEP2.format(clauses=clauses, contract=rec["context"]))
    out = {}
    for name in present:
        a = reply.get(name) if isinstance(reply.get(name), dict) else {}
        spans = a.get("spans") or []
        spans = [spans] if isinstance(spans, str) else [s for s in spans if isinstance(s, str) and s.strip()]
        out[name] = {"spans": list(dict.fromkeys(spans)), "confidence": conf(a.get("confidence")) if spans else 0.0}
    return out


STEP3 = """Below is a contract cut into four equal parts, each marked with a header such as <<< REGION 25-50% >>>.
For each clause you get the passage(s) that were quoted as its answer. Say in which region each passage sits.

Reply with ONE JSON object and nothing else:
{{"<clause name>": {{"regions": ["0-25%" | "25-50%" | "50-75%" | "75-100%", ...], "confidence": <0..1>}}, ...}}

- "regions": one entry per quoted passage, in the same order as the passages are listed. Use the region where the passage starts.
- "confidence": how sure you are of the regions you gave, from 0 to 1. Lower it if you could not find a passage.

Clauses and their quoted passages:
{items}

--- CONTRACT, IN FOUR REGIONS ---
{contract}
"""

def step3_region(rec: dict, where: dict) -> dict:
    """Step 3: in which quarter of the contract is the answer? -> {clause: {"regions": [...], "confidence": float}}"""
    todo = {n: w for n, w in where.items() if w["spans"]}
    if not todo:
        return {}
    text, q = rec["context"], len(rec["context"]) // 4
    parts = [text[:q], text[q: 2 * q], text[2 * q: 3 * q], text[3 * q:]]
    marked = "\n".join(f"\n<<< REGION {REGIONS[i]} >>>\n{part}" for i, part in enumerate(parts))
    items = "\n\n".join(f"[{n}]\n" + "\n".join(f"  passage {i}: {s}" for i, s in enumerate(w["spans"], 1)) for n, w in todo.items())
    reply = ask(STEP3.format(items=items, contract=marked))
    out = {}
    for name in todo:
        a = reply.get(name) if isinstance(reply.get(name), dict) else {}
        regions = a.get("regions") or []
        regions = [regions] if isinstance(regions, str) else [str(r) for r in regions]
        out[name] = {"regions": regions, "confidence": conf(a.get("confidence"))}
    return out


STEP4 = """For each clause below you get a question about a contract and the answer that was quoted from the contract.
Decide what KIND of answer the question needs.

- "data": the answer is a specific value that can be picked out and counted: a date, a name of a person or company,
  a place, an amount of money, a number, a duration, a document title.
- "semantic": the answer is a provision that has to be read and understood; it cannot be reduced to one value.

Reply with ONE JSON object and nothing else:
{{"<clause name>": {{"kind": "data" or "semantic",
                    "entity_type": "DATE | PERSON | ORG | GPE | MONEY | DURATION | QUANTITY | DOC | LAW | none",
                    "value": "<for data: the key value exactly as it is written in the contract, several values separated by ' | '; for semantic: empty>",
                    "occurrences": <for data: how many times that value occurs in the whole contract (for several values, the rarest one); for semantic: 0>,
                    "reasoning": "<one sentence: for semantic, why this passage answers the question; for data, why this value is the answer>",
                    "confidence": <0..1>}}, ...}}

- "confidence": how sure you are that the quoted answer correctly answers the question, given the kind of answer it is, from 0 to 1.
  For data, a value that occurs once and is clearly labelled deserves more confidence than one of several competing values.
  For semantic, be strict: is this the passage a lawyer would highlight, and is anything missing?

Clauses:
{items}

--- CONTRACT ---
{contract}
"""

def step4_kind(rec: dict, where: dict) -> dict:
    """Step 4: semantic answer or data? -> {clause: {"kind", "entity_type", "value", "occurrences", "reasoning", "confidence"}}"""
    todo = {n: w for n, w in where.items() if w["spans"]}
    if not todo:
        return {}
    items = "\n\n".join(f"[{n}]\nQuestion: {rec['questions'][n]}\nAnswer: {' | '.join(w['spans'])}" for n, w in todo.items())
    reply = ask(STEP4.format(items=items, contract=rec["context"]))
    out = {}
    for name in todo:
        a = reply.get(name) if isinstance(reply.get(name), dict) else {}
        kind = "data" if str(a.get("kind", "")).lower() == "data" else "semantic"
        try:
            claimed = int(a.get("occurrences") or 0)
        except (TypeError, ValueError):
            claimed = 0
        out[name] = {"kind": kind, "entity_type": str(a.get("entity_type") or "none"), "value": str(a.get("value") or "") if kind == "data" else "",
                     "occurrences": claimed, "reasoning": str(a.get("reasoning") or ""), "confidence": conf(a.get("confidence"))}
    return out

## The function: `agent_confidence(rec)`

It runs the four steps in order, checks what code can check, and combines the four confidences.

**The four confidences, per clause**

| | Starts as | Adjusted by code |
| --- | --- | --- |
| `c1` present | step 1's confidence in "present" | none |
| `c2` where | step 2's confidence in its quotes | multiplied by the share of quotes that are really in the contract |
| `c3` region | step 3's confidence in its regions | set to 0.2 if a region it named is not where the quote actually sits |
| `c4` kind | step 4's confidence that the answer fits the question | for data, set to 0.1 if a value it named does not occur in the contract at all |

**The final confidence** is the geometric mean of the four: `(c1 × c2 × c3 × c4) ^ (1/4)`.
A geometric mean is used so that one weak step pulls the result down hard (a plain average would hide it),
and any step at 0 makes the final 0.

A clause gets **no answer** (null) when step 1 says it is absent, when step 2 finds nothing to quote, or when none of the
quotes can be found in the contract. The answer text itself is always the contract's own wording, not the model's retyped copy.

In [4]:
def agent_confidence(rec: dict, verbose: bool = True) -> tuple[dict, pd.DataFrame]:
    """Runs the four-step agent on one CUAD record.

    Returns (prediction, details):
      prediction  {"doc_id": ..., "fields": {clause: {"value": ..., "confidence": final}}}  in the scorer's format
      details     one row per clause with every step's output and confidence
    """
    text = rec["context"]
    squashed, pos = squash(text)
    say = print if verbose else (lambda *a, **k: None)

    s1 = step1_is_present(rec)
    present = [n for n, v in s1.items() if v["present"]]
    say(f"step 1  is the answer present?   {len(present)} of {len(s1)} clauses judged present")

    s2 = step2_where(rec, present)
    say(f"step 2  where is it?             {sum(bool(v['spans']) for v in s2.values())} clauses quoted, "
        f"{sum(len(v['spans']) for v in s2.values())} passages")

    s3 = step3_region(rec, s2)
    say(f"step 3  which quarter?           {len(s3)} clauses placed")

    s4 = step4_kind(rec, s2)
    say(f"step 4  semantic or data?        {sum(v['kind'] == 'data' for v in s4.values())} data, "
        f"{sum(v['kind'] == 'semantic' for v in s4.values())} semantic")

    fields, rows = {}, []
    for name in rec["questions"]:
        row = {"clause": name, "present": s1[name]["present"], "c1 present": s1[name]["confidence"]}
        spans = s2.get(name, {}).get("spans", [])
        found = [(s, hit) for s, hit in ((s, locate(s, squashed, pos)) for s in spans) if hit is not None]
        if not s1[name]["present"] or not found:
            fields[name] = {"value": None}
            rows.append({**row, "answer": None, "final confidence": None})
            continue

        # step 2, checked: how many of the quotes are really in the contract
        c2 = s2[name]["confidence"] * len(found) / len(spans)

        # step 3, checked: do the regions the agent named match where the quotes sit
        actual = [region_of(start, len(text)) for _, (start, _) in found]
        claimed = s3.get(name, {}).get("regions", [])
        regions_ok = set(actual) <= set(claimed) and bool(claimed)
        c3 = s3.get(name, {}).get("confidence", 0.0) if regions_ok else 0.2

        # step 4, checked: for data answers, count the value in the contract
        k = s4.get(name, {"kind": "semantic", "entity_type": "none", "value": "", "occurrences": 0, "reasoning": "", "confidence": 0.0})
        parts = [squash(v)[0].strip() for v in k["value"].split("|") if v.strip()] if k["kind"] == "data" else []
        counted = min(squashed.count(v) for v in parts) if parts else None       # several values: the rarest one
        c4 = 0.1 if counted == 0 else k["confidence"]

        c1 = s1[name]["confidence"]
        final = round((c1 * c2 * c3 * c4) ** 0.25, 3)

        # the answer uses the contract's own text for each quote that was found
        value = " | ".join(text[start:end] for _, (start, end) in found)
        fields[name] = {"value": value, "confidence": final}
        rows.append({**row, "answer": " ".join(value.split()), "quotes found": f"{len(found)}/{len(spans)}", "c2 where": round(c2, 3),
                     "region (agent)": ", ".join(claimed), "region (actual)": ", ".join(actual), "c3 region": round(c3, 3),
                     "kind": k["kind"], "entity": k["entity_type"] if k["kind"] == "data" else "", "value": k["value"],
                     "occurs (agent)": k["occurrences"] if k["kind"] == "data" else None, "occurs (counted)": counted,
                     "c4 kind": round(c4, 3), "reasoning": k["reasoning"], "final confidence": final})

    return {"doc_id": rec["doc_id"], "fields": fields}, pd.DataFrame(rows).set_index("clause")

## Run it on one record

Change the index (0 to 380) or pass `doc_id=` to try another contract.

In [5]:
rec = load_record(0)
print(rec["doc_id"])
print(f"context: {len(rec['context']):,} characters, {len(rec['questions'])} questions\n")

prediction, details = agent_confidence(rec)

cuad:LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT
context: 54,290 characters, 41 questions

step 1  is the answer present?   22 of 41 clauses judged present
step 2  where is it?             21 clauses quoted, 28 passages
step 3  which quarter?           21 clauses placed
step 4  semantic or data?        5 data, 16 semantic


### Step by step: the four confidences and the final one

One row per clause the agent answered. Clauses it judged absent are left out here.

In [6]:
answered = details[details["final confidence"].notna()]
answered[["c1 present", "c2 where", "c3 region", "c4 kind", "final confidence", "answer"]].sort_values("final confidence", ascending=False)

,c1 present,c2 where,c3 region,c4 kind,final confidence,answer
clause,,,,,,
Agreement Date,0.96,0.900,0.97,0.93,0.940,"7th day of September, 1999"
Governing Law,0.98,0.950,0.97,0.85,0.936,This Agreement is to be construed according to the laws of the Sta...
Insurance,0.96,0.900,0.96,0.92,0.935,Company will carry a reasonable amount of product liability insura...
No-Solicit Of Employees,0.95,0.900,0.95,0.93,0.932,During the Term of this Agreement and for a period of twelve (12) ...
Warranty Duration,0.97,0.900,0.95,0.90,0.929,Company further warrants that the Products sold hereunder shall be...
Covenant Not To Sue,0.93,0.900,0.95,0.92,0.925,"During the Term of this Agreement and for three years thereafter, ..."
Document Name,0.97,0.900,0.97,0.85,0.921,DISTRIBUTOR AGREEMENT
Anti-Assignment,0.96,0.900,0.96,0.85,0.916,No assignment of this Agreement or any right accruing hereunder sh...
Parties,0.98,0.850,0.97,0.85,0.910,Electric City Corp. | Electric City of Illinois LLC


### Step 3 in detail: where in the contract

`region (agent)` is what the agent said; `region (actual)` is computed from the character position of each quote.
When they disagree, `c3` is set to 0.2.

In [7]:
answered[["quotes found", "region (agent)", "region (actual)", "c3 region"]]

,quotes found,region (agent),region (actual),c3 region
clause,,,,
Document Name,1/1,0-25%,0-25%,0.97
Parties,2/2,"0-25%, 0-25%","0-25%, 0-25%",0.97
Agreement Date,1/1,0-25%,0-25%,0.97
Effective Date,1/1,50-75%,50-75%,0.93
Expiration Date,1/1,0-25%,0-25%,0.95
Renewal Term,1/1,0-25%,0-25%,0.95
Governing Law,1/1,75-100%,75-100%,0.97
Exclusivity,3/3,"0-25%, 0-25%, 75-100%","0-25%, 0-25%, 75-100%",0.90
Competitive Restriction Exception,2/2,"0-25%, 75-100%","0-25%, 75-100%",0.92


### Step 4 in detail: semantic answer or data

For data answers, `occurs (agent)` is the count the agent gave and `occurs (counted)` is the real number of times the value appears in the contract
(when the agent named several values, the count of the rarest one).

In [8]:
answered[["kind", "entity", "value", "occurs (agent)", "occurs (counted)", "c4 kind", "reasoning"]]

,kind,entity,value,occurs (agent),occurs (counted),c4 kind,reasoning
clause,,,,,,,
Document Name,data,DOC,DISTRIBUTOR AGREEMENT,3.0,3.0,0.85,The title of the contract is the document name; it appears as the ...
Parties,data,ORG,Electric City Corp. | Electric City of Illinois LLC,2.0,1.0,0.85,The two signing companies named in the preamble and signature bloc...
Agreement Date,data,DATE,"7th day of September, 1999",1.0,1.0,0.93,The preamble states the agreement is made on this date.
Effective Date,semantic,,,NaN,NaN,0.70,The passage says the agreement is effective immediately subject to...
Expiration Date,semantic,,,NaN,NaN,0.70,"The term is ten years starting from delivery of the last Sample, s..."
Renewal Term,semantic,,,NaN,NaN,0.85,The renewal clause is conditioned on Distributor's compliance and ...
Governing Law,data,GPE,State of Illinois,3.0,2.0,0.85,The governing law clause names Illinois as the state whose law gov...
Exclusivity,semantic,,,NaN,NaN,0.85,"The exclusive appointment, the sole-source purchase obligation and..."
Competitive Restriction Exception,semantic,,,NaN,NaN,0.75,The passages are carve-outs: freedom to sell in markets without an...


## How does the final confidence behave against gold?

Gold comes in only here. Outcomes use the official scorer's rule (`span_f1` from `5_score.py`, correct at 0.8 or more).
The last column shows what accepting every answer down to that row would save or cost: +1 for a correct answer, -100 for a wrong or hallucinated one.

This is one contract, so it shows how the function works, not how well it works.

In [7]:
spec = importlib.util.spec_from_file_location("score", ROOT / "5_score.py")
score = importlib.util.module_from_spec(spec); spec.loader.exec_module(score)

def outcome(pred, gold):
    if pred is None:
        return ("missed" if gold is not None else "true null"), None
    if gold is None:
        return "hallucinated", None
    f1 = score.span_f1(pred, gold)
    return ("correct" if f1 >= score.SPAN_F1_THRESHOLD else "wrong"), round(f1, 2)

rows = []
for name, f in prediction["fields"].items():
    o, f1 = outcome(f["value"], rec["gold"][name])
    rows.append({"clause": name, "outcome": o, "token F1": f1, "final confidence": f.get("confidence")})
compare = pd.DataFrame(rows).set_index("clause")
print(compare["outcome"].value_counts().to_string(), "\n")

ranked = compare[compare["final confidence"].notna()].sort_values("final confidence", ascending=False).join(
    details[["c1 present", "c2 where", "c3 region", "c4 kind", "kind"]])
ranked["net saving if accepted down to here"] = ranked["outcome"].eq("correct").map({True: 1, False: -100}).cumsum()
ranked[["final confidence", "c1 present", "c2 where", "c3 region", "c4 kind", "kind", "outcome", "token F1", "net saving if accepted down to here"]]

outcome
true null       20
correct          9
wrong            9
hallucinated     2
missed           1 



,final confidence,c1 present,c2 where,c3 region,c4 kind,kind,outcome,token F1,net saving if accepted down to here
clause,,,,,,,,,
Agreement Date,0.940,0.96,0.900,0.97,0.93,data,correct,0.80,1
Governing Law,0.936,0.98,0.950,0.97,0.85,data,correct,1.00,2
Insurance,0.935,0.96,0.900,0.96,0.92,semantic,wrong,0.73,-98
No-Solicit Of Employees,0.932,0.95,0.900,0.95,0.93,semantic,correct,1.00,-97
Warranty Duration,0.929,0.97,0.900,0.95,0.90,data,wrong,0.15,-197
Covenant Not To Sue,0.925,0.93,0.900,0.95,0.92,semantic,correct,1.00,-196
Document Name,0.921,0.97,0.900,0.97,0.85,data,correct,1.00,-195
Anti-Assignment,0.916,0.96,0.900,0.96,0.85,semantic,wrong,0.60,-295
Parties,0.910,0.98,0.850,0.97,0.85,data,wrong,0.64,-395


## The prediction line

The same record in the format `5_score.py` and the hidden test expect, with the final confidence as `confidence`.

In [8]:
line = json.dumps(prediction, ensure_ascii=False)
print(f"{len(prediction['fields'])} fields, {sum(f['value'] is not None for f in prediction['fields'].values())} answered\n")
print(line[:1200] + " ...")

41 fields, 20 answered

{"doc_id": "cuad:LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT", "fields": {"Document Name": {"value": "DISTRIBUTOR AGREEMENT", "confidence": 0.921}, "Parties": {"value": "Electric City Corp. | Electric City of Illinois LLC", "confidence": 0.91}, "Agreement Date": {"value": "7th day of September, 1999", "confidence": 0.94}, "Effective Date": {"value": "this  Agreement,  subject to the  commencement  date                   established  in Section 1.3,  shall be effective  immediately.", "confidence": 0.714}, "Expiration Date": {"value": "The term of this  Agreement  shall be ten (10)                            years (the \"Term\")  which shall  commence on the date                            upon which the Company  delivers to  Distributor  the                            last Sample, as defined  hereinafter.", "confidence": 0.727}, "Renewal Term": {"value": "If Distributor                            complies with all of the terms of this Agreement, the     